# Bank Client Segmentation: Phase 2 - Dimensionality Reduction

This notebook follows the Exploratory Data Analysis (EDA) phase. Here we will preprocess the clean data, compute the Gower distance matrix, and perform dimensionality reduction (PCA, MCA, FAMD, t-SNE, UMAP) to visually identify clusters before running K-Medoids.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import gower
from sklearn.manifold import TSNE
import umap
from sklearn.decomposition import PCA
import prince
import warnings
warnings.filterwarnings('ignore')

# Load the data
path = Path("Data") / "Dataset1_BankClients.xlsx"
data = pd.read_excel(path)

# Drop ID column
if 'ID' in data.columns:
    data = data.drop(columns=['ID'])

print(f"Data shape: {data.shape}")
data.head()

## 1. Data Preprocessing

We need to split the data into numerical and categorical features. Then we will scale the numerical features using `MinMaxScaler` and One-Hot Encode the categorical features to create our final feature matrix `X`.

In [ ]:
# Define feature types
categorical_cols = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']
numerical_cols = [c for c in data.columns if c not in categorical_cols]

# Scale Numerical Features
scaler = MinMaxScaler()
data_scaled_num = pd.DataFrame(scaler.fit_transform(data[numerical_cols]), columns=numerical_cols)

# One-Hot Encode Categorical Features
data_encoded_cat = pd.get_dummies(data[categorical_cols].astype(str), drop_first=True)

# Combine into final feature matrix X
X = pd.concat([data_scaled_num, data_encoded_cat], axis=1)

print(f"Numerical features shape: {data_scaled_num.shape}")
print(f"Categorical (encoded) features shape: {data_encoded_cat.shape}")
print(f"Final Feature Matrix X shape: {X.shape}")

## 2. Gower Distance Matrix

Since our data is a mix of continuous and categorical variables, standard Euclidean distance is inappropriate. We compute the Gower distance matrix, which perfectly handles mixed data types and outputs a pairwise distance between 0 (identical) and 1 (maximally different).

In [ ]:
# Compute Gower distance matrix
# Note: gower_matrix can take a dataframe with mixed types directly. 
# It handles scaling and categorical differences automatically.
# We pass the original data (before manual scaling/encoding) to let gower do its magic accurately!

print("Computing Gower Distance Matrix... this may take a moment.")
gower_dist_matrix = gower.gower_matrix(data)
print(f"Gower Matrix Shape: {gower_dist_matrix.shape}")

## 3. Linear Dimensionality Reduction

Linear methods try to preserve the global structure and variance of the data. We will use:
- **PCA:** For numerical data only.
- **MCA:** For categorical data only.
- **FAMD (Factor Analysis of Mixed Data):** The gold standard linear method that combines PCA and MCA for mixed data.

In [ ]:
# 1. PCA (Numerical Only)
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(data_scaled_num)

# 2. MCA (Categorical Only)
mca = prince.MCA(n_components=2, random_state=42)
mca_result = mca.fit_transform(data[categorical_cols].astype(str))

# 3. FAMD (Mixed Data)
famd = prince.FAMD(n_components=2, n_iter=3, random_state=42)
famd_result = famd.fit_transform(data)

# Store results for plotting
linear_results = {
    'PCA (Numerical)': pca_result,
    'MCA (Categorical)': mca_result.values,
    'FAMD (Mixed Data)': famd_result.values
}

## 4. Non-Linear Dimensionality Reduction (Manifold Learning)

Non-linear methods are excellent at unfolding complex structures and preserving local neighborhoods, making them perfect for visualizing clusters. We will use:
- **t-SNE:** The classic standard for visualizing clusters.
- **UMAP:** The modern standard, often faster and preserves global structure better than t-SNE.

**CRITICAL:** We pass `metric='precomputed'` and feed them the Gower Distance Matrix so they respect the mixed data types!

In [ ]:
# 1. t-SNE (Precomputed Gower Distance)
tsne = TSNE(n_components=2, metric='precomputed', init='random', random_state=42, n_jobs=-1)
tsne_result = tsne.fit_transform(gower_dist_matrix)

# 2. UMAP (Precomputed Gower Distance)
umap_model = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
umap_result = umap_model.fit_transform(gower_dist_matrix)

# Store results for plotting
nonlinear_results = {
    't-SNE (Gower Distance)': tsne_result,
    'UMAP (Gower Distance)': umap_result
}

## 5. Visual Comparison

Let's plot all 5 projections side-by-side. We are looking for visible groupings, islands, or blobs that indicate natural 'personas' in the data. The UMAP and t-SNE plots powered by the Gower distance matrix usually provide the clearest picture for clustering.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Plot Linear Methods
for i, (title, result) in enumerate(linear_results.items()):
    ax = axes[i]
    sns.scatterplot(x=result[:, 0], y=result[:, 1], alpha=0.5, s=20, color='royalblue', ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

# Plot Non-Linear Methods
for i, (title, result) in enumerate(nonlinear_results.items()):
    ax = axes[i+3] # Shift to the second row
    sns.scatterplot(x=result[:, 0], y=result[:, 1], alpha=0.5, s=20, color='darkorange', ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

# Hide the last empty subplot
axes[5].axis('off')

plt.tight_layout()
plt.show()